In [ ]:
import torch
from utils import *
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

In [ ]:
# Data parameters
srgan_checkpoint = "./checkpoints/checkpoint_srgan_best.pth.tar"
HR_image_path = "data/benchmark/Set14/HR/baboon.png"
LR_image_path = "data/benchmark/Set14/LR_bicubic/X4/baboonx4.png"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

In [ ]:
# Load models
srgan_generator = torch.load(srgan_checkpoint, weights_only=False)["generator"].to(device)
srgan_generator.eval()

In [ ]:
hr_img = Image.open(HR_image_path, mode="r")
hr_img = hr_img.convert("RGB")
lr_img = Image.open(LR_image_path, mode="r")
lr_img = lr_img.convert("RGB")

In [ ]:
# Bicubic Upsampling
bicubic_img = lr_img.resize((hr_img.width, hr_img.height), Image.BICUBIC)
# Super-resolution (SR) with SRGAN
sr_img_srgan = srgan_generator(
    convert_image(lr_img, source="pil", target="imagenet-norm", device=device)
    .unsqueeze(0)
    .to(device)
)
sr_img_srgan = sr_img_srgan.squeeze(0).detach()
sr_img_srgan = convert_image(
    sr_img_srgan, source="[-1, 1]", target="pil", device=device
)

In [ ]:
# Visualization
plt.figure("Bicubic")
plt.imshow(bicubic_img)
plt.axis("off")
plt.title("Bicubic")
plt.figure("SRGAN")
plt.imshow(sr_img_srgan)
plt.axis("off")
plt.title("SRGAN")
plt.figure("Original HR")
plt.imshow(hr_img)
plt.axis("off")
plt.title("Original HR")
plt.show()